# 03 — Android malware classifier

This notebook is the primary environment for exploring, preparing, splitting, training, evaluating, and comparing the Android permission/attribute classifier. It reads the source workbook and processed candidate CSV without changing either.

The OUTPUT interpretation used here is provisional: 0 = non-malware and 1 = malware, corroborated by the project-provided external report with the same count distribution but not confirmed by the dataset publisher. These are source labels, not independently verified labels. Any model result is limited by that provenance and the dataset collection.

## A — Environment and reproducibility

Use the active backend ML environment; do not install or upgrade packages from this notebook. Fixed seed: 20261026. Expensive cells report progress and elapsed time. No code cell has been run while creating this notebook.

In [2]:
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime, timezone
import csv, hashlib, json, math, os, platform, posixpath, random, re, shutil, sys, tempfile, time, zipfile
from xml.etree import ElementTree as ET

import joblib
import numpy as np
import pandas as pd
import scipy
import sklearn
from scipy import sparse
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, confusion_matrix, f1_score, precision_score,
    recall_score, roc_auc_score
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline

SEED = 20261026
random.seed(SEED)
np.random.seed(SEED)
ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "backend" / "data" / "processed" / "android_permission_candidates.csv").is_file():
    ROOT = ROOT.parent
if not (ROOT / "backend" / "data" / "processed" / "android_permission_candidates.csv").is_file():
    raise RuntimeError("Start Jupyter from the repository or one of its directories.")

PROCESSED = ROOT / "backend" / "data" / "processed"
CANDIDATE_CSV = PROCESSED / "android_permission_candidates.csv"
AUDIT_JSON = PROCESSED / "android_feature_audit.json"
CONFLICT_CSV = PROCESSED / "android_package_label_conflicts.csv"
HEADER_TXT = PROCESSED / "android_column_headers.txt"
RAW_ZIP = ROOT / "backend" / "data" / "raw" / "Android Benign and Malware Dataset2.zip"
WORKBOOK = ROOT / "backend" / "data" / "raw" / "android_extracted" / "Android Benign and Malware Dataset2" / "AndroidDataset.xlsx"
MODEL_DIR = ROOT / "backend" / "models" / "android"
REPORT_DIR = ROOT / "backend" / "reports" / "ml" / "android_malware"
SEED = 20261026
print("Existing model output directory:", [p.name for p in MODEL_DIR.iterdir()] if MODEL_DIR.is_dir() else "absent")
print("Existing report output directory:", [p.name for p in REPORT_DIR.iterdir()] if REPORT_DIR.is_dir() else "absent")

print({
    "python": sys.version, "executable": sys.executable,
    "numpy": np.__version__, "pandas": pd.__version__, "scipy": scipy.__version__,
    "scikit_learn": sklearn.__version__, "joblib": joblib.__version__,
    "seed": SEED,
    "dataset_paths": {"candidate_csv": str(CANDIDATE_CSV), "raw_zip": str(RAW_ZIP),
                      "read_only_workbook": str(WORKBOOK)},
    "sizes_bytes": {str(p): p.stat().st_size if p.is_file() else None
                    for p in (CANDIDATE_CSV, AUDIT_JSON, CONFLICT_CSV, HEADER_TXT, RAW_ZIP, WORKBOOK)},
})

Existing model output directory: absent
Existing report output directory: absent
{'python': '3.14.6 (tags/v3.14.6:c63aec6, Jun 10 2026, 10:26:10) [MSC v.1944 64 bit (AMD64)]', 'executable': 'c:\\Users\\Sushanth CD\\AppData\\Local\\Programs\\Python\\Python314\\python.exe', 'numpy': '2.4.4', 'pandas': '3.0.2', 'scipy': '1.17.1', 'scikit_learn': '1.8.0', 'joblib': '1.5.3', 'seed': 20261026, 'dataset_paths': {'candidate_csv': 'C:\\Projects\\cyber-security-awareness-platform\\backend\\data\\processed\\android_permission_candidates.csv', 'raw_zip': 'C:\\Projects\\cyber-security-awareness-platform\\backend\\data\\raw\\Android Benign and Malware Dataset2.zip', 'read_only_workbook': 'C:\\Projects\\cyber-security-awareness-platform\\backend\\data\\raw\\android_extracted\\Android Benign and Malware Dataset2\\AndroidDataset.xlsx'}, 'sizes_bytes': {'C:\\Projects\\cyber-security-awareness-platform\\backend\\data\\processed\\android_permission_candidates.csv': 358164107, 'C:\\Projects\\cyber-security

## B — Load and inspect the data

The processed candidate CSV contains source_row_index, label_original, source_dataset, review fields, and 1,427 feature fields. It does not contain Package or Category. Do not assume row order is the join key: reconstruct Package, Category, and OUTPUT from the read-only workbook by its Excel source row index, then verify every row and label explicitly.

In [3]:
with AUDIT_JSON.open(encoding="utf-8") as f:
    audit = json.load(f)
candidate_columns = pd.read_csv(CANDIDATE_CSV, nrows=0).columns.tolist()
candidate_sample = pd.read_csv(CANDIDATE_CSV, nrows=2, dtype=str, keep_default_na=False)
feature_columns = candidate_columns[5:]
required_metadata = ["source_row_index", "label_original", "source_dataset", "review_required", "review_reason"]
assert candidate_columns[:5] == required_metadata
assert "Package" not in candidate_columns and "Category" not in candidate_columns
assert feature_columns == audit["feature_columns"]
assert len(feature_columns) == 1427
print({
    "candidate_csv_columns": len(candidate_columns),
    "candidate_feature_columns": len(feature_columns),
    "metadata_columns": candidate_columns[:5],
    "label_present": "label_original" in candidate_columns,
    "package_identifier_present": "Package" in candidate_columns,
    "category_present": "Category" in candidate_columns,
    "source_row_id_present": "source_row_index" in candidate_columns,
    "sample_source_rows": candidate_sample["source_row_index"].tolist(),
    "sample_labels": candidate_sample["label_original"].tolist(),
    "audit_rows": audit["data_rows"],
    "audit_output_distribution": audit["output_distribution"],
    "audit_output_semantics_verified": audit["output_semantics_verified"],
})

with zipfile.ZipFile(RAW_ZIP, "r") as archive:
    archive_entries = [{"name": x.filename, "compressed_bytes": x.compress_size,
                        "uncompressed_bytes": x.file_size}
                       for x in archive.infolist()]
print("Raw archive entries:", archive_entries)
workbook_temp_dir = None
if not WORKBOOK.is_file():
    with zipfile.ZipFile(RAW_ZIP, "r") as archive:
        matches = [n for n in archive.namelist() if n.endswith("/AndroidDataset.xlsx")]
        if len(matches) != 1:
            raise FileNotFoundError(f"Expected one AndroidDataset.xlsx in archive; found {matches}")
        workbook_temp_dir = tempfile.TemporaryDirectory(prefix="android-workbook-readonly-")
        WORKBOOK = Path(workbook_temp_dir.name) / "AndroidDataset.xlsx"
        with archive.open(matches[0]) as source, WORKBOOK.open("wb") as destination:
            shutil.copyfileobj(source, destination, length=8 * 1024 * 1024)
        print("Stream-extracted workbook to temporary storage; raw ZIP remains unchanged.")
with zipfile.ZipFile(WORKBOOK, "r") as book:
    workbook_members = book.namelist()
    workbook_xml = ET.fromstring(book.read("xl/workbook.xml"))
    workbook_sheets = [e.attrib.get("name") for e in workbook_xml.iter()
                       if e.tag.rsplit("}", 1)[-1] == "sheet"]
print("Workbook sheets:", workbook_sheets)
print("Workbook OpenXML members:", [n for n in workbook_members
                                    if n.startswith(("xl/workbook", "xl/worksheets/", "docProps/"))][:20])
print("Preparation report says the only sheet is Sheet1, dimension A1:BBZ121190, with 121,189 data rows.")

{'candidate_csv_columns': 1432, 'candidate_feature_columns': 1427, 'metadata_columns': ['source_row_index', 'label_original', 'source_dataset', 'review_required', 'review_reason'], 'label_present': True, 'package_identifier_present': False, 'category_present': False, 'source_row_id_present': True, 'sample_source_rows': ['2', '3'], 'sample_labels': ['0', '0'], 'audit_rows': 121189, 'audit_output_distribution': {'0': 111190, '1': 9999}, 'audit_output_semantics_verified': False}
Raw archive entries: [{'name': 'Android Benign and Malware Dataset2/AndroidDataset.xlsx', 'compressed_bytes': 288990005, 'uncompressed_bytes': 518057984}]
Workbook sheets: ['Sheet1']
Workbook OpenXML members: ['xl/workbook.xml', 'xl/worksheets/sheet1.xml', 'docProps/core.xml', 'docProps/app.xml']
Preparation report says the only sheet is Sheet1, dimension A1:BBZ121190, with 121,189 data rows.


### Label semantics and target definition

The workbook's Mendeley description and local metadata do not document OUTPUT polarity. The project-provided external research report describes the same 111,190 / 9,999 distribution as non-malware / malware. This supports, but does not publisher-confirm, the provisional mapping below. The notebook retains label_original unchanged and marks results as provisional; it does not independently verify any app label.

In [4]:
SOURCE_TO_TARGET = {"0": 0, "1": 1}
TARGET_NAMES = {0: "non-malware (provisional)", 1: "malware (provisional)"}
TARGET_MAPPING_NOTE = (
    "Provisional OUTPUT mapping: 0=non-malware, 1=malware. Distribution is externally corroborated "
    "as described in project context, but the dataset publisher does not document polarity; labels "
    "are not independently verified."
)
TARGET_THRESHOLD = 0.5
print(TARGET_MAPPING_NOTE)

Provisional OUTPUT mapping: 0=non-malware, 1=malware. Distribution is externally corroborated as described in project context, but the dataset publisher does not document polarity; labels are not independently verified.


## C — Reconstruct and verify row-level provenance

The preparation script writes source_row_index as the original Excel worksheet row number and writes label_original from that same row's OUTPUT cell while deliberately omitting Package and Category. This makes a keyed reconstruction possible. The standard-library streaming parser below reads only Package, Category, and OUTPUT from each workbook row; it does not load the 494+ MB workbook into Pandas.

In [5]:
MAIN_NS = "http://schemas.openxmlformats.org/spreadsheetml/2006/main"
DOC_REL_NS = "http://schemas.openxmlformats.org/officeDocument/2006/relationships"
PKG_REL_NS = "http://schemas.openxmlformats.org/package/2006/relationships"

def local_name(tag):
    return tag.rsplit("}", 1)[-1]

def excel_column_index(reference):
    match = re.match(r"[A-Z]+", reference)
    if not match:
        raise ValueError(f"Invalid worksheet cell reference: {reference!r}")
    index = 0
    for letter in match.group(0):
        index = index * 26 + ord(letter) - ord("A") + 1
    return index - 1

def workbook_identity_rows(path):
    identities = {}
    with zipfile.ZipFile(path, "r") as book:
        workbook_xml = ET.fromstring(book.read("xl/workbook.xml"))
        relationships = ET.fromstring(book.read("xl/_rels/workbook.xml.rels"))
        targets = {
            rel.attrib["Id"]: rel.attrib["Target"]
            for rel in relationships.findall(f"{{{PKG_REL_NS}}}Relationship")
        }
        sheet = workbook_xml.find(f"{{{MAIN_NS}}}sheets/{{{MAIN_NS}}}sheet")
        if sheet is None:
            raise ValueError("Workbook does not contain a worksheet.")
        sheet_name = sheet.attrib["name"]
        target = targets[sheet.attrib[f"{{{DOC_REL_NS}}}id"]]
        target = target.lstrip("/")
        sheet_path = posixpath.normpath(target if target.startswith("xl/") else posixpath.join("xl", target))

        shared_strings = []
        with book.open("xl/sharedStrings.xml") as stream:
            for _, item in ET.iterparse(stream, events=("end",)):
                if local_name(item.tag) == "si":
                    shared_strings.append("".join(item.itertext()))
                    item.clear()

        header = None
        observed_dimension = None
        with book.open(sheet_path) as stream:
            for event, element in ET.iterparse(stream, events=("start", "end")):
                if event == "start" and local_name(element.tag) == "dimension":
                    observed_dimension = element.attrib.get("ref")
                if event != "end" or local_name(element.tag) != "row":
                    continue
                row_number = int(element.attrib["r"])
                cells = {}
                for item in element:
                    if local_name(item.tag) != "c":
                        continue
                    col = excel_column_index(item.attrib.get("r", ""))
                    if row_number != 1 and col not in {0, 1, 1429}:
                        continue
                    value_node = next((v for v in item if local_name(v.tag) == "v"), None)
                    value = value_node.text if value_node is not None and value_node.text else ""
                    if item.attrib.get("t") == "s" and value:
                        value = shared_strings[int(value)]
                    elif item.attrib.get("t") == "inlineStr":
                        value = "".join(item.itertext())
                    cells[col] = value
                if row_number == 1:
                    header = [cells.get(i, "") for i in range(max(cells) + 1)]
                    if header[0] != "Package" or header[1] != "Category" or header[1429] != "OUTPUT":
                        raise ValueError("Workbook header does not match the audited Package/Category/OUTPUT schema.")
                else:
                    identities[row_number] = {
                        "package": cells.get(0, ""),
                        "category": cells.get(1, ""),
                        "output": cells.get(1429, ""),
                    }
                element.clear()
    return sheet_name, observed_dimension, header, identities

scan_started = time.perf_counter()
sheet_name, worksheet_dimension, workbook_headers, identity_by_row = workbook_identity_rows(WORKBOOK)
expected_workbook_headers = ["Package", "Category", *feature_columns, "OUTPUT"]
if workbook_headers != expected_workbook_headers:
    raise ValueError("Original workbook header sequence differs from processed audit/candidate features.")
documented_headers = []
for line in HEADER_TXT.read_text(encoding="utf-8-sig").splitlines()[1:]:
    fields = line.split("\t", 2)
    if len(fields) != 3:
        raise ValueError(f"Malformed documented header line: {line[:100]!r}")
    documented_headers.append(fields[2])
if documented_headers != workbook_headers:
    raise ValueError("android_column_headers.txt does not match original workbook header order.")
print({
    "workbook_scan_seconds": round(time.perf_counter() - scan_started, 2),
    "worksheet": sheet_name, "dimension": worksheet_dimension,
    "header_columns": len(workbook_headers),
    "workbook_data_rows": len(identity_by_row),
    "first_source_row": min(identity_by_row), "last_source_row": max(identity_by_row),
    "unique_nonempty_package_ids": len({v["package"] for v in identity_by_row.values() if v["package"]}),
})

{'workbook_scan_seconds': 2982.98, 'worksheet': 'Sheet1', 'dimension': 'A1:BBZ121190', 'header_columns': 1430, 'workbook_data_rows': 121189, 'first_source_row': 2, 'last_source_row': 121190, 'unique_nonempty_package_ids': 47891}


## D — Quality audit and feature preparation

Scan the processed CSV in bounded chunks. Verify binary/numeric completeness instead of assuming the permission fields are binary. Recompute exact feature-vector hashes, constant columns, and duplicate columns from observed candidate values and compare the findings with the prepared audit. Remove constant and redundant feature columns deterministically, retaining the first eligible column in original workbook order.

In [6]:
chunk_size = 512
row_ids, labels_original, review_flags, source_names = [], [], [], set()
feature_hashes = []
column_chunk_bits = [[] for _ in feature_columns]
column_min = np.full(len(feature_columns), 255, dtype=np.uint8)
column_max = np.zeros(len(feature_columns), dtype=np.uint8)
scan_started = time.perf_counter()
scanned_rows = 0
for chunk_number, chunk in enumerate(pd.read_csv(
    CANDIDATE_CSV, chunksize=chunk_size, dtype=str, keep_default_na=False
), start=1):
    ids = chunk["source_row_index"].astype(np.int64).to_numpy()
    labels = chunk["label_original"].to_numpy()
    if not set(labels).issubset(SOURCE_TO_TARGET):
        raise ValueError(f"Unexpected OUTPUT labels in chunk {chunk_number}: {set(labels)}")
    if not set(chunk["review_required"]).issubset({"true", "false"}):
        raise ValueError("Unexpected review_required encoding.")
    values = chunk[feature_columns].to_numpy(dtype=str)
    if values.size and not np.isin(values, ["0", "1"]).all():
        bad = sorted(set(values.reshape(-1)) - {"0", "1"})
        raise ValueError(f"Feature values are missing or nonbinary: {bad[:10]}")
    binary = values.astype(np.uint8, copy=False)
    row_ids.extend(ids.tolist())
    labels_original.extend(labels.tolist())
    review_flags.extend(v == "true" for v in chunk["review_required"])
    source_names.update(chunk["source_dataset"].unique().tolist())
    feature_hashes.extend(
        hashlib.sha256(packed.tobytes()).hexdigest()
        for packed in np.packbits(binary, axis=1, bitorder="little")
    )
    column_min = np.minimum(column_min, binary.min(axis=0))
    column_max = np.maximum(column_max, binary.max(axis=0))
    packed_columns = np.packbits(binary, axis=0, bitorder="little")
    for j, packed_column in enumerate(packed_columns.T):
        column_chunk_bits[j].append(packed_column.tobytes())
    scanned_rows += len(chunk)
    if chunk_number % 40 == 0:
        print(f"Scanned {scanned_rows:,} candidate rows in {time.perf_counter() - scan_started:.1f}s")

if len(set(row_ids)) != len(row_ids):
    raise ValueError("Processed candidate source_row_index values are not unique.")
if set(source_names) != {"android_benign_malware_dataset2"}:
    raise ValueError(f"Unexpected source_dataset values: {source_names}")
if scanned_rows != audit["data_rows"]:
    raise ValueError(f"Candidate row count {scanned_rows} != audit row count {audit['data_rows']}")

column_signatures = [
    hashlib.sha256(b"".join(pieces)).hexdigest() for pieces in column_chunk_bits
]
constant_columns = {
    name for name, lo, hi in zip(feature_columns, column_min, column_max) if lo == hi
}
signature_groups = defaultdict(list)
for name, signature in zip(feature_columns, column_signatures):
    signature_groups[signature].append(name)
duplicate_column_groups = [names for names in signature_groups.values() if len(names) > 1]
duplicate_columns_beyond_first = sum(len(names) - 1 for names in duplicate_column_groups)
seen_signatures = set()
retained_features = []
for name, signature in zip(feature_columns, column_signatures):
    if name in constant_columns or signature in seen_signatures:
        continue
    seen_signatures.add(signature)
    retained_features.append(name)

assert constant_columns == set(audit["constant_feature_columns"])
assert duplicate_columns_beyond_first == audit["duplicate_feature_columns_beyond_first"]
assert len(retained_features) == len(feature_columns) - len(constant_columns) - sum(
    len(names) - 1 for names in duplicate_column_groups
    if not all(name in constant_columns for name in names)
)
labels_array = np.asarray([SOURCE_TO_TARGET[x] for x in labels_original], dtype=np.int8)
overall_target_counts = Counter(labels_array.tolist())
overall_target_shares = {str(k): round(v / len(labels_array), 4) for k, v in overall_target_counts.items()}
row_ids_array = np.asarray(row_ids, dtype=np.int64)
feature_hash_array = np.asarray(feature_hashes, dtype=object)
print({
    "rows": scanned_rows,
    "source_label_counts": dict(Counter(labels_original)),
    "provisional_target_class_imbalance": {"counts": dict(overall_target_counts),
        "shares": {str(k): v / len(labels_array) for k, v in overall_target_counts.items()}},
    "target_counts_provisional": dict(Counter(labels_array.tolist())),
    "review_required_rows": int(sum(review_flags)),
    "missing_feature_values": 0,
    "nonbinary_feature_columns": 0,
    "feature_columns_original": len(feature_columns),
    "constant_columns_removed": len(constant_columns),
    "duplicate_columns_beyond_first": duplicate_columns_beyond_first,
    "feature_columns_retained": len(retained_features),
    "unique_feature_vectors": len(set(feature_hashes)),
    "feature_vector_duplicate_rows_beyond_first": scanned_rows - len(set(feature_hashes)),
    "scan_seconds": round(time.perf_counter() - scan_started, 2),
})

Scanned 20,480 candidate rows in 90.7s
Scanned 40,960 candidate rows in 183.7s
Scanned 61,440 candidate rows in 273.5s
Scanned 81,920 candidate rows in 364.3s
Scanned 102,400 candidate rows in 432.7s
{'rows': 121189, 'source_label_counts': {'0': 111190, '1': 9999}, 'provisional_target_class_imbalance': {'counts': {0: 111190, 1: 9999}, 'shares': {'0': 0.9174925116966062, '1': 0.08250748830339387}}, 'target_counts_provisional': {0: 111190, 1: 9999}, 'review_required_rows': 121189, 'missing_feature_values': 0, 'nonbinary_feature_columns': 0, 'feature_columns_original': 1427, 'constant_columns_removed': 6, 'duplicate_columns_beyond_first': 1200, 'feature_columns_retained': 226, 'unique_feature_vectors': 12222, 'feature_vector_duplicate_rows_beyond_first': 108967, 'scan_seconds': 479.22}


## E — Label alignment, package conflicts, and duplicate policy

Verify source-row keys and each label against the workbook before any split or model fit. Join app identity only for audit/grouping, never as a predictive feature.

Predeclared policy:
- Build connected components from shared nonempty Package identifiers and identical full feature vectors. Feature identity is a leakage-control relation only; it does not assert that different apps with identical permissions are the same app.
- Quarantine a whole connected component if any package identifier has conflicting OUTPUT labels or any identical feature vector has conflicting labels. Keep source labels and all rows in the audit manifest.
- Collapse exact complete duplicate records to one modeling observation, but keep every source row in the manifest and keep the component intact across partitions.
- Do not resolve or relabel conflicts automatically.

In [7]:
align_started = time.perf_counter()
if len(identity_by_row) != len(row_ids):
    raise ValueError("Workbook and processed candidate row counts differ.")
if set(identity_by_row) != set(row_ids):
    missing = set(row_ids) - set(identity_by_row)
    extra = set(identity_by_row) - set(row_ids)
    raise ValueError(f"Source-row key mismatch: candidate-only={len(missing)}, workbook-only={len(extra)}")
alignment_mismatches = []
packages, categories, workbook_outputs = [], [], []
for source_row, original_label in zip(row_ids, labels_original):
    source = identity_by_row[source_row]
    packages.append(source["package"])
    categories.append(source["category"])
    workbook_outputs.append(source["output"])
    if str(source["output"]) != str(original_label):
        alignment_mismatches.append(source_row)
if alignment_mismatches:
    raise ValueError(f"Processed label differs from workbook OUTPUT at rows {alignment_mismatches[:10]}")

package_labels = defaultdict(set)
package_feature_vectors = defaultdict(set)
feature_labels = defaultdict(set)
exact_record_rows = defaultdict(list)
for i, (package, category, label, vector_hash) in enumerate(
    zip(packages, categories, labels_original, feature_hashes)
):
    if package:
        package_labels[package].add(label)
        package_feature_vectors[package].add(vector_hash)
    feature_labels[vector_hash].add(label)
    record_key = hashlib.sha256(json.dumps(
        [package, category, label, vector_hash], ensure_ascii=False, separators=(",", ":")
    ).encode("utf-8")).hexdigest()
    exact_record_rows[record_key].append(i)

conflicting_packages = {p for p, ys in package_labels.items() if len(ys) > 1}
package_row_counts = Counter(packages)
record_key_by_index = {i: key for key, indices in exact_record_rows.items() for i in indices}
conflicting_feature_hashes = {h for h, ys in feature_labels.items() if len(ys) > 1}
with CONFLICT_CSV.open(encoding="utf-8", newline="") as stream:
    conflict_inventory = list(csv.DictReader(stream))
conflict_file_packages = {row["package_id_not_for_model"] for row in conflict_inventory}
assert conflicting_packages == conflict_file_packages, "Reconstructed package conflicts differ from the saved conflict inventory."
assert len(conflict_inventory) == audit["package_identifiers_with_conflicting_OUTPUT"]
assert len(alignment_mismatches) == 0
exact_duplicate_rows_beyond_first = sum(len(v) - 1 for v in exact_record_rows.values())
if exact_duplicate_rows_beyond_first != audit["full_row_duplicates_beyond_first"]:
    raise ValueError("Recomputed exact-record duplicate count differs from the saved workbook audit.")
if scanned_rows - len(set(feature_hashes)) != audit["candidate_feature_only_duplicate_rows_beyond_first"]:
    raise ValueError("Recomputed repeated-feature-vector count differs from the saved workbook audit.")
print({
    "rows_joined_by_source_row_index": len(row_ids),
    "workbook_candidate_label_mismatches": len(alignment_mismatches),
    "package_ids": len(package_labels),
    "repeated_package_ids": sum(count > 1 for count in package_row_counts.values()),
    "package_ids_with_multiple_feature_vectors": sum(len(v) > 1 for v in package_feature_vectors.values()),
    "conflicting_package_ids_with_multiple_feature_vectors": sum(len(package_feature_vectors[p]) > 1 for p in conflicting_packages),
    "conflicting_package_ids_with_one_feature_vector": sum(len(package_feature_vectors[p]) == 1 for p in conflicting_packages),
    "conflicting_package_ids": len(conflicting_packages),
    "conflicting_feature_vectors": len(conflicting_feature_hashes),
    "exact_complete_duplicate_rows_beyond_first": exact_duplicate_rows_beyond_first,
    "audit_full_row_duplicates_beyond_first": audit["full_row_duplicates_beyond_first"],
    "conflict_inventory_rows": len(conflict_inventory),
    "alignment_seconds": round(time.perf_counter() - align_started, 2),
})

{'rows_joined_by_source_row_index': 121189, 'workbook_candidate_label_mismatches': 0, 'package_ids': 47891, 'repeated_package_ids': 38868, 'package_ids_with_multiple_feature_vectors': 5098, 'conflicting_package_ids_with_multiple_feature_vectors': 3526, 'conflicting_package_ids_with_one_feature_vector': 147, 'conflicting_package_ids': 3673, 'conflicting_feature_vectors': 326, 'exact_complete_duplicate_rows_beyond_first': 65851, 'audit_full_row_duplicates_beyond_first': 65851, 'conflict_inventory_rows': 3673, 'alignment_seconds': 5.51}


## F — Connected components and exclusions

Use union-find over original source records, joining shared package IDs and identical feature-vector hashes. Component-level quarantine propagates conflict reasons across every linked record. Report component sizes, mixed-label components, class counts, duplicates, and exclusions before assigning partitions.

In [8]:
class DisjointSet:
    def __init__(self, size):
        self.parent = list(range(size))
        self.rank = [0] * size
    def find(self, x):
        if self.parent[x] != x:
            self.parent[x] = self.find(self.parent[x])
        return self.parent[x]
    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return
        if self.rank[ra] < self.rank[rb]:
            ra, rb = rb, ra
        self.parent[rb] = ra
        if self.rank[ra] == self.rank[rb]:
            self.rank[ra] += 1

dsu = DisjointSet(len(row_ids))
first_package, first_feature = {}, {}
for i, (package, vector_hash) in enumerate(zip(packages, feature_hashes)):
    if package:
        if package in first_package:
            dsu.union(i, first_package[package])
        else:
            first_package[package] = i
    if vector_hash in first_feature:
        dsu.union(i, first_feature[vector_hash])
    else:
        first_feature[vector_hash] = i

members = defaultdict(list)
for i in range(len(row_ids)):
    members[dsu.find(i)].append(i)
component_for_row = {}
component_ids = {}
component_reasons = defaultdict(set)
row_reasons = [set() for _ in row_ids]
for i, (package, vector_hash) in enumerate(zip(packages, feature_hashes)):
    if package in conflicting_packages:
        row_reasons[i].add("package_identifier_has_conflicting_OUTPUT")
    if vector_hash in conflicting_feature_hashes:
        row_reasons[i].add("identical_feature_vector_has_conflicting_labels")
for root, indices in members.items():
    component_id = hashlib.sha256(
        "\\n".join(str(row_ids[i]) for i in sorted(indices, key=lambda j: row_ids[j])).encode("utf-8")
    ).hexdigest()
    component_ids[root] = component_id
    for i in indices:
        component_for_row[i] = component_id
        component_reasons[component_id].update(row_reasons[i])

quarantined_components = {
    component_ids[root] for root in members if component_reasons[component_ids[root]]
}
component_row_counts = {component_ids[root]: len(indices) for root, indices in members.items()}
propagated_exclusion_counts = {
    reason: sum(size for cid, size in component_row_counts.items() if reason in component_reasons[cid])
    for reason in {r for reasons in component_reasons.values() for r in reasons}
}
quarantine_mask = np.asarray(
    [component_for_row[i] in quarantined_components for i in range(len(row_ids))], dtype=bool
)
component_sizes = [len(v) for v in members.values()]
component_label_counts = []
for root, indices in members.items():
    component_label_counts.append(dict(Counter(labels_array[indices].tolist())))
mixed_label_components = sum(len(counts) > 1 for counts in component_label_counts)
component_label_pattern_counts = Counter(tuple(sorted(counts.items())) for counts in component_label_counts)
percentile = lambda values, q: int(np.quantile(values, q, method="higher")) if values else None
reason_record_counts = Counter(reason for reasons in row_reasons for reason in reasons)
unique_record_indices = [indices[0] for indices in exact_record_rows.values()]
duplicate_representative_indices = set(unique_record_indices)
print({
    "records": len(row_ids),
    "unique_feature_vectors": len(set(feature_hashes)),
    "connected_components": len(members),
    "component_size": {"median": percentile(component_sizes, .5),
                       "p90": percentile(component_sizes, .9),
                       "p95": percentile(component_sizes, .95),
                       "maximum": max(component_sizes)},
    "largest_components": sorted(component_sizes, reverse=True)[:10],
    "components_with_both_labels": mixed_label_components,
    "component_label_count_patterns": {str(k): v for k, v in component_label_pattern_counts.items()},
    "label_counts_by_component_examples": component_label_counts[:10],
    "quarantined_components": len(quarantined_components),
    "quarantined_source_records": int(quarantine_mask.sum()),
    "excluded_by_reason_record_counts_overlap_possible": dict(reason_record_counts),
    "component_propagated_exclusion_record_counts_overlap_possible": propagated_exclusion_counts,
    "complete_duplicate_rows_collapsed_for_modeling": exact_duplicate_rows_beyond_first,
    "unique_complete_records": len(unique_record_indices),
    "eligible_source_records": int((~quarantine_mask).sum()),
    "eligible_unique_complete_records": sum(
        not quarantine_mask[i] for i in unique_record_indices
    ),
})

{'records': 121189, 'unique_feature_vectors': 12222, 'connected_components': 7256, 'component_size': {'median': 2, 'p90': 9, 'p95': 16, 'maximum': 84932}, 'largest_components': [84932, 270, 252, 249, 190, 180, 180, 152, 131, 129], 'components_with_both_labels': 999, 'component_label_count_patterns': {'((0, 2),)': 2595, '((0, 79221), (1, 5711))': 1, '((0, 70), (1, 1))': 1, '((0, 4),)': 827, '((0, 4), (1, 1))': 106, '((0, 16), (1, 3))': 4, '((0, 26),)': 4, '((0, 47),)': 1, '((0, 6),)': 278, '((0, 14),)': 29, '((0, 24), (1, 2))': 1, '((0, 43),)': 2, '((0, 7),)': 40, '((0, 12),)': 41, '((0, 168), (1, 12))': 1, '((0, 23),)': 5, '((0, 34), (1, 11))': 1, '((0, 6), (1, 1))': 57, '((0, 5),)': 85, '((0, 8),)': 141, '((0, 28),)': 5, '((0, 17), (1, 4))': 2, '((0, 10), (1, 1))': 18, '((0, 10), (1, 2))': 7, '((0, 24),)': 5, '((0, 21), (1, 4))': 1, '((0, 113), (1, 4))': 1, '((0, 14), (1, 1))': 3, '((0, 8), (1, 2))': 19, '((0, 177), (1, 3))': 1, '((0, 89),)': 1, '((0, 5), (1, 1))': 19, '((0, 2), (1, 2

## F — Leakage-resistant 70/15/15 split

StratifiedGroupKFold receives per-source-record labels and connected-component group IDs, so stratification considers individual record labels while components remain indivisible. Twenty folds are allocated as 14 train folds, 3 validation folds, and 3 test folds. This approximates 70/15/15; large components and class imbalance may force deviations. Conflicted components are excluded before splitting.

In [9]:
split_started = time.perf_counter()
eligible_source_indices = np.flatnonzero(~quarantine_mask)
eligible_groups = np.asarray([component_for_row[i] for i in eligible_source_indices], dtype=object)
eligible_y = labels_array[eligible_source_indices]
if len(set(eligible_groups)) < 20:
    raise ValueError("Fewer than 20 eligible components; the planned group-stratified allocation is infeasible.")
splitter = StratifiedGroupKFold(n_splits=20, shuffle=True, random_state=SEED)
folds = list(splitter.split(np.zeros(len(eligible_source_indices)), eligible_y, groups=eligible_groups))
fold_partition = {**{i: "test" for i in range(3)},
                  **{i: "validation" for i in range(3, 6)},
                  **{i: "train" for i in range(6, 20)}}
component_partition = {}
for fold_index, (_, held_positions) in enumerate(folds):
    partition = fold_partition[fold_index]
    for component_id in set(eligible_groups[held_positions].tolist()):
        if component_id in component_partition:
            raise AssertionError("A connected component appeared in multiple held folds.")
        component_partition[component_id] = partition
if set(component_partition) != set(eligible_groups.tolist()):
    raise AssertionError("Some eligible components were not allocated.")

row_partition = [
    "quarantined" if quarantine_mask[i] else component_partition[component_for_row[i]]
    for i in range(len(row_ids))
]
parts = ("train", "validation", "test")
partition_record_sets = {p: set() for p in parts}
partition_package_sets = defaultdict(set)
partition_feature_sets = defaultdict(set)
for i, partition in enumerate(row_partition):
    if partition == "quarantined":
        continue
    partition_record_sets[partition].add(row_ids[i])
    if packages[i]:
        partition_package_sets[packages[i]].add(partition)
    partition_feature_sets[feature_hashes[i]].add(partition)

assert all(len(v) == 1 for v in partition_package_sets.values())
assert all(len(v) == 1 for v in partition_feature_sets.values())
assert all(
    len({row_partition[i] for i in indices}) == 1
    for indices in members.values()
)
assert all(row_partition[i] == "quarantined" for i in range(len(row_ids)) if quarantine_mask[i])
assert not any(p in {"train", "validation", "test"} for p in row_partition if p == "quarantined")

split_verified = True
unique_by_partition = {
    p: [i for i in unique_record_indices if row_partition[i] == p]
    for p in parts
}
source_counts = {
    p: {"rows": int(sum(v == p for v in row_partition)),
        "label_counts": dict(Counter(labels_array[np.asarray([i for i,v in enumerate(row_partition) if v == p],dtype=int)].tolist()))}
    for p in parts
}
eligible_rows = int((~quarantine_mask).sum())
split_summary = {
    p: {**source_counts[p],
        "percent_of_eligible_source_rows": 100 * source_counts[p]["rows"] / eligible_rows,
        "unique_complete_records_for_model": len(unique_by_partition[p]),
        "unique_record_label_counts": dict(Counter(labels_array[unique_by_partition[p]].tolist()))}
    for p in parts
}
print({
    "target_percent": {"train": 70, "validation": 15, "test": 15},
    "achieved": split_summary,
    "deviation_percentage_points": {p: split_summary[p]["percent_of_eligible_source_rows"] - target
                                    for p, target in zip(parts, [70,15,15])},
    "without_package_identifier_rows": sum(not p for p in packages),
    "package_isolation": all(len(v) == 1 for v in partition_package_sets.values()),
    "feature_vector_isolation": all(len(v) == 1 for v in partition_feature_sets.values()),
    "component_isolation": True,
    "quarantine_isolation": True,
    "split_validation_passed": split_verified,
    "split_seconds": round(time.perf_counter() - split_started, 2),
})

{'target_percent': {'train': 70, 'validation': 15, 'test': 15}, 'achieved': {'train': {'rows': 15367, 'label_counts': {0: 13987, 1: 1380}, 'percent_of_eligible_source_rows': 69.99316784331587, 'unique_complete_records_for_model': 7321, 'unique_record_label_counts': {0: 6039, 1: 1282}}, 'validation': {'rows': 3294, 'label_counts': {0: 2997, 1: 297}, 'percent_of_eligible_source_rows': 15.003416078342063, 'unique_complete_records_for_model': 1577, 'unique_record_label_counts': {0: 1301, 1: 276}}, 'test': {'rows': 3294, 'label_counts': {0: 2998, 1: 296}, 'percent_of_eligible_source_rows': 15.003416078342063, 'unique_complete_records_for_model': 1609, 'unique_record_label_counts': {0: 1327, 1: 282}}}, 'deviation_percentage_points': {'train': -0.006832156684126289, 'validation': 0.0034160783420631446, 'test': 0.0034160783420631446}, 'without_package_identifier_rows': 0, 'package_isolation': True, 'feature_vector_isolation': True, 'component_isolation': True, 'quarantine_isolation': True, 'sp

## G — Build model matrix after split checks pass

Only the retained binary feature columns enter X. Package, Category, OUTPUT, source row IDs, review fields, and all other audit metadata are excluded. Build the reduced matrix in chunks and keep it sparse. The model fit cell refuses to run unless all row alignment, conflict, and leakage assertions above completed.

In [10]:
if not split_verified:
    raise RuntimeError("Split validation did not pass; refusing to construct a training matrix.")
matrix_started = time.perf_counter()
sparse_chunks = []
matrix_row_ids = []
use_columns = required_metadata + retained_features
for chunk_number, chunk in enumerate(pd.read_csv(
    CANDIDATE_CSV, usecols=use_columns, chunksize=chunk_size,
    dtype=str, keep_default_na=False
), start=1):
    matrix_row_ids.extend(chunk["source_row_index"].astype(np.int64).tolist())
    values = chunk[retained_features].to_numpy(dtype=np.uint8)
    sparse_chunks.append(sparse.csr_matrix(values))
    if chunk_number % 40 == 0:
        print(f"Prepared sparse feature chunks covering {len(matrix_row_ids):,} rows")
X_all = sparse.vstack(sparse_chunks, format="csr")
if matrix_row_ids != row_ids:
    raise ValueError("Reduced feature matrix row order does not match the verified row ID sequence.")
if X_all.shape != (len(row_ids), len(retained_features)):
    raise ValueError(f"Unexpected design matrix shape: {X_all.shape}")
print({
    "matrix_shape": X_all.shape,
    "format": X_all.getformat(),
    "nonzero_values": int(X_all.nnz),
    "density": float(X_all.nnz / (X_all.shape[0] * X_all.shape[1])),
    "feature_columns_retained": retained_features,
    "matrix_build_seconds": round(time.perf_counter() - matrix_started, 2),
})

Prepared sparse feature chunks covering 20,480 rows
Prepared sparse feature chunks covering 40,960 rows
Prepared sparse feature chunks covering 61,440 rows
Prepared sparse feature chunks covering 81,920 rows
Prepared sparse feature chunks covering 102,400 rows
{'matrix_shape': (121189, 226), 'format': 'csr', 'nonzero_values': 673757, 'density': 0.024599804138303098, 'feature_columns_retained': ['Default : Access DRM content. (S)', 'Default : Access Email provider data (S)', 'Default : Access all system downloads (S)', 'Default : Access download manager. (S)', 'Default : Advanced download manager functions. (S)', 'Default : Audio File Access (S)', 'Default : Install DRM content. (S)', 'Default : Modify Google service configuration (S)', 'Default : Modify Google settings (S)', 'Default : Move application resources (S)', 'Default : Read Google settings (S)', 'Default : Send download notifications. (S)', 'Default : Voice Search Shortcuts (S)', 'Default : access SurfaceFlinger (S)', 'Defaul

## G — Logistic Regression baseline and class-weight comparison

Fit ordinary and class-weight-balanced Logistic Regression on the same training components. Use validation F1 at a fixed 0.5 threshold to select the Logistic Regression configuration, with validation recall as a tie-break. Do not tune or select with test results.

In [11]:
if not split_verified:
    raise RuntimeError("Refusing to train before split verification.")
THRESHOLD = 0.5
train_idx = np.asarray(unique_by_partition["train"], dtype=int)
validation_idx = np.asarray(unique_by_partition["validation"], dtype=int)
test_idx = np.asarray(unique_by_partition["test"], dtype=int)
y_train, y_validation, y_test = labels_array[train_idx], labels_array[validation_idx], labels_array[test_idx]

def make_logistic(class_weight):
    return Pipeline([("classifier", LogisticRegression(
        C=1.0, solver="liblinear", max_iter=2000,
        class_weight=class_weight, random_state=SEED,
    ))])

def calculate_metrics(y_true, scores, threshold=THRESHOLD):
    pred = (np.asarray(scores) >= threshold).astype(np.int8)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0,1]).ravel()
    return {
        "records": int(len(y_true)), "tn": int(tn), "fp": int(fp),
        "fn": int(fn), "tp": int(tp),
        "precision": precision_score(y_true, pred, zero_division=0),
        "recall": recall_score(y_true, pred, zero_division=0),
        "f1": f1_score(y_true, pred, zero_division=0),
        "specificity": tn / (tn + fp) if tn + fp else None,
        "accuracy": accuracy_score(y_true, pred),
        "roc_auc": roc_auc_score(y_true, scores) if len(set(y_true)) == 2 else None,
    }

logistic_models, logistic_validation = {}, {}
for weight in (None, "balanced"):
    started = time.perf_counter()
    model = make_logistic(weight)
    model.fit(X_all[train_idx], y_train)
    scores = model.predict_proba(X_all[validation_idx])[:, list(model.classes_).index(1)]
    logistic_models[str(weight)] = model
    logistic_validation[str(weight)] = calculate_metrics(y_validation, scores)
    print("Logistic class_weight=", weight, "validation=", logistic_validation[str(weight)],
          "elapsed_seconds=", round(time.perf_counter() - started, 2))

selected_logistic_key = max(
    logistic_validation,
    key=lambda key: (logistic_validation[key]["f1"], logistic_validation[key]["recall"])
)
logistic_model = logistic_models[selected_logistic_key]
print("Selected Logistic Regression by validation F1, recall tie-break:", selected_logistic_key)

Logistic class_weight= None validation= {'records': 1577, 'tn': 1297, 'fp': 4, 'fn': 31, 'tp': 245, 'precision': 0.9839357429718876, 'recall': 0.8876811594202898, 'f1': 0.9333333333333333, 'specificity': np.float64(0.9969254419677172), 'accuracy': 0.9778059606848446, 'roc_auc': 0.9701093918836123} elapsed_seconds= 0.16
Logistic class_weight= balanced validation= {'records': 1577, 'tn': 1261, 'fp': 40, 'fn': 20, 'tp': 256, 'precision': 0.8648648648648649, 'recall': 0.927536231884058, 'f1': 0.8951048951048951, 'specificity': np.float64(0.9692544196771714), 'accuracy': 0.9619530754597336, 'roc_auc': 0.9731755951386336} elapsed_seconds= 0.16
Selected Logistic Regression by validation F1, recall tie-break: None


## H — Random Forest baseline

Train one bounded Random Forest configuration on the identical training partition and reduced feature matrix. No hyperparameter search is performed. Both final models will be evaluated once on the same held-out test components.

In [12]:
if not split_verified:
    raise RuntimeError("Refusing to train before split verification.")
rf_started = time.perf_counter()
random_forest_model = Pipeline([("classifier", RandomForestClassifier(
    n_estimators=200, max_features="sqrt", min_samples_leaf=2,
    class_weight="balanced_subsample", random_state=SEED,
    n_jobs=-1, max_samples=None,
))])
random_forest_model.fit(X_all[train_idx], y_train)
print({
    "training_records": len(train_idx),
    "random_forest_config": {"n_estimators": 200, "max_features": "sqrt",
                             "min_samples_leaf": 2, "class_weight": "balanced_subsample",
                             "random_state": SEED, "n_jobs": -1},
    "elapsed_seconds": round(time.perf_counter() - rf_started, 2),
})

{'training_records': 7321, 'random_forest_config': {'n_estimators': 200, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'class_weight': 'balanced_subsample', 'random_state': 20261026, 'n_jobs': -1}, 'elapsed_seconds': 2.85}


## I — Held-out evaluation and error analysis

The provisional positive class is malware (OUTPUT=1). Report confusion counts and precision, recall, F1, specificity, accuracy, and ROC-AUC when both classes are present. Report source-row and unique-complete-record class counts for each partition. False negatives mean missed malware under the provisional labels; false positives mean non-malware flagged as malware. Both costs matter, and dataset labels are not independently verified.

In [13]:
def class_scores(model, X):
    classes = list(model.classes_)
    return model.predict_proba(X)[:, classes.index(1)]

lr_test_scores = class_scores(logistic_model, X_all[test_idx])
rf_test_scores = class_scores(random_forest_model, X_all[test_idx])
lr_test_metrics = calculate_metrics(y_test, lr_test_scores)
rf_test_metrics = calculate_metrics(y_test, rf_test_scores)
lr_validation_scores = class_scores(logistic_model, X_all[validation_idx])
lr_selected_validation_metrics = calculate_metrics(y_validation, lr_validation_scores)
rf_validation_scores = class_scores(random_forest_model, X_all[validation_idx])
rf_validation_metrics = calculate_metrics(y_validation, rf_validation_scores)
comparison = {
    "logistic_regression": lr_test_metrics,
    "random_forest": rf_test_metrics,
}
print("Partition counts:", json.dumps(split_summary, indent=2))
print("Test comparison:", json.dumps(comparison, indent=2))
comparison_table = pd.DataFrame([
    {"model": name, **{key: metrics[key] for key in ("precision", "recall", "f1", "specificity", "accuracy", "roc_auc")}}
    for name, metrics in comparison.items()
])
display(comparison_table)
higher_f1_model = comparison_table.sort_values(["f1", "recall"], ascending=False).iloc[0]["model"]
f1_difference = abs(lr_test_metrics["f1"] - rf_test_metrics["f1"])
comparison_interpretation = f"Higher test F1 in this split: {higher_f1_model}; absolute F1 difference={f1_difference:.4f}. This descriptive difference is not a significance test or evidence of real-world superiority."
print(comparison_interpretation)
print("Logistic validation:", json.dumps(lr_selected_validation_metrics, indent=2))
print("Random Forest validation:", json.dumps(rf_validation_metrics, indent=2))

test_predictions = []
for position, source_index in enumerate(test_idx):
    lr_score = float(lr_test_scores[position])
    rf_score = float(rf_test_scores[position])
    test_predictions.append({
        "source_row_index": int(row_ids[source_index]),
        "label_original": labels_original[source_index],
        "target_interpretation": TARGET_NAMES[int(labels_array[source_index])],
        "component_id": component_for_row[source_index],
        "logistic_regression_score": lr_score,
        "logistic_regression_prediction": int(lr_score >= THRESHOLD),
        "random_forest_score": rf_score,
        "random_forest_prediction": int(rf_score >= THRESHOLD),
    })
prediction_frame = pd.DataFrame(test_predictions)
error_frame = prediction_frame[
    (prediction_frame["logistic_regression_prediction"] != y_test) |
    (prediction_frame["random_forest_prediction"] != y_test)
].copy()
print({
    "test_prediction_rows": len(prediction_frame),
    "rows_with_any_model_error": len(error_frame),
    "logistic_false_positives": lr_test_metrics["fp"],
    "logistic_false_negatives": lr_test_metrics["fn"],
    "random_forest_false_positives": rf_test_metrics["fp"],
    "random_forest_false_negatives": rf_test_metrics["fn"],
})

Partition counts: {
  "train": {
    "rows": 15367,
    "label_counts": {
      "0": 13987,
      "1": 1380
    },
    "percent_of_eligible_source_rows": 69.99316784331587,
    "unique_complete_records_for_model": 7321,
    "unique_record_label_counts": {
      "0": 6039,
      "1": 1282
    }
  },
  "validation": {
    "rows": 3294,
    "label_counts": {
      "0": 2997,
      "1": 297
    },
    "percent_of_eligible_source_rows": 15.003416078342063,
    "unique_complete_records_for_model": 1577,
    "unique_record_label_counts": {
      "0": 1301,
      "1": 276
    }
  },
  "test": {
    "rows": 3294,
    "label_counts": {
      "0": 2998,
      "1": 296
    },
    "percent_of_eligible_source_rows": 15.003416078342063,
    "unique_complete_records_for_model": 1609,
    "unique_record_label_counts": {
      "0": 1327,
      "1": 282
    }
  }
}
Test comparison: {
  "logistic_regression": {
    "records": 1609,
    "tn": 1315,
    "fp": 12,
    "fn": 32,
    "tp": 250,
    "precision"

,model,precision,recall,f1,specificity,accuracy,roc_auc
0,logistic_regression,0.954198,0.886525,0.919118,0.990957,0.972654,0.986705
1,random_forest,0.932862,0.936170,0.934513,0.985682,0.977004,0.989696


Higher test F1 in this split: random_forest; absolute F1 difference=0.0154. This descriptive difference is not a significance test or evidence of real-world superiority.
Logistic validation: {
  "records": 1577,
  "tn": 1297,
  "fp": 4,
  "fn": 31,
  "tp": 245,
  "precision": 0.9839357429718876,
  "recall": 0.8876811594202898,
  "f1": 0.9333333333333333,
  "specificity": 0.9969254419677172,
  "accuracy": 0.9778059606848446,
  "roc_auc": 0.9701093918836123
}
Random Forest validation: {
  "records": 1577,
  "tn": 1281,
  "fp": 20,
  "fn": 25,
  "tp": 251,
  "precision": 0.9261992619926199,
  "recall": 0.9094202898550725,
  "f1": 0.9177330895795247,
  "specificity": 0.9846272098385856,
  "accuracy": 0.9714648065948003,
  "roc_auc": 0.9869832570263677
}
{'test_prediction_rows': 1609, 'rows_with_any_model_error': 59, 'logistic_false_positives': 12, 'logistic_false_negatives': 32, 'random_forest_false_positives': 19, 'random_forest_false_negatives': 18}


## J — Model comparison and feature interpretation

Compare held-out metrics on the same test components. The Logistic Regression class-weight setting was chosen using validation only. A metric difference is descriptive of this dataset and split; it is not evidence of real-world superiority. Coefficients and tree importance indicate associations in this feature matrix, not that a permission causes malware.

In [14]:
lr_estimator = logistic_model.named_steps["classifier"]
rf_estimator = random_forest_model.named_steps["classifier"]
lr_coefficients = lr_estimator.coef_[0]
lr_order = np.argsort(np.abs(lr_coefficients))[::-1][:20]
rf_importances = rf_estimator.feature_importances_
rf_order = np.argsort(rf_importances)[::-1][:20]
importance_rows = []
for rank, idx in enumerate(lr_order, start=1):
    importance_rows.append({"model": "Logistic Regression", "rank": rank,
                            "feature": retained_features[idx],
                            "importance": float(lr_coefficients[idx]),
                            "direction": "malware-associated" if lr_coefficients[idx] > 0 else "non-malware-associated"})
for rank, idx in enumerate(rf_order, start=1):
    importance_rows.append({"model": "Random Forest", "rank": rank,
                            "feature": retained_features[idx],
                            "importance": float(rf_importances[idx]),
                            "direction": "impurity importance; direction not defined"})
importance_frame = pd.DataFrame(importance_rows)
display(importance_frame)
print("Interpretation caution: feature importance is associational and does not establish causation.")

,model,rank,feature,importance,direction
0,Logistic Regression,1,Default : write contact data (S),6.079661,malware-associated
1,Logistic Regression,2,Default : Audio File Access (S),5.395318,malware-associated
2,Logistic Regression,3,Your accounts : contacts data in Google accoun...,4.442584,malware-associated
3,Logistic Regression,4,Your accounts : access all Google services (S),4.367612,malware-associated
4,Logistic Regression,5,System tools : format external storage (D),4.141217,malware-associated
5,Logistic Regression,6,Your personal information : write Browser's hi...,4.109353,malware-associated
6,Logistic Regression,7,Default : modify the Google services map (S),3.639987,malware-associated
7,Logistic Regression,8,Your personal information : write contact data...,3.632693,malware-associated
8,Logistic Regression,9,Default : enable or disable application compon...,3.303531,malware-associated
9,Logistic Regression,10,Your personal information : read Browser's his...,-3.004145,non-malware-associated


Interpretation caution: feature importance is associational and does not establish causation.


## K — Save new artifacts after all checks

All paths below are separate from the phishing model and reports. The split-validation guard must remain true. The notebook refuses to overwrite any target file. Outputs include fitted pipelines, configuration, all-row split manifest, review/exclusion inventory, metrics, predictions/errors, and a Markdown report. The manifest retains labels and identifiers for audit; neither identifiers nor Category are model features.

In [15]:
if not split_verified:
    raise RuntimeError("No model artifacts will be saved because split validation failed.")
MODEL_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_PATHS = {
    "logistic_model": MODEL_DIR / "android_logistic_regression_grouped.joblib",
    "random_forest_model": MODEL_DIR / "android_random_forest_grouped.joblib",
    "config": REPORT_DIR / "experiment_config.json",
    "metrics": REPORT_DIR / "metrics.json",
    "report": REPORT_DIR / "experiment_report.md",
    "manifest": REPORT_DIR / "split_manifest.csv",
    "review": REPORT_DIR / "data_review.csv",
    "predictions": REPORT_DIR / "test_predictions.csv",
    "errors": REPORT_DIR / "test_errors.csv",
}
existing_targets = [str(path) for path in ARTIFACT_PATHS.values() if path.exists()]
if existing_targets:
    raise FileExistsError("Refusing to overwrite existing artifacts: " + ", ".join(existing_targets))
print("Validated new output targets:", {key: str(path) for key, path in ARTIFACT_PATHS.items()})

Validated new output targets: {'logistic_model': 'C:\\Projects\\cyber-security-awareness-platform\\backend\\models\\android\\android_logistic_regression_grouped.joblib', 'random_forest_model': 'C:\\Projects\\cyber-security-awareness-platform\\backend\\models\\android\\android_random_forest_grouped.joblib', 'config': 'C:\\Projects\\cyber-security-awareness-platform\\backend\\reports\\ml\\android_malware\\experiment_config.json', 'metrics': 'C:\\Projects\\cyber-security-awareness-platform\\backend\\reports\\ml\\android_malware\\metrics.json', 'report': 'C:\\Projects\\cyber-security-awareness-platform\\backend\\reports\\ml\\android_malware\\experiment_report.md', 'manifest': 'C:\\Projects\\cyber-security-awareness-platform\\backend\\reports\\ml\\android_malware\\split_manifest.csv', 'review': 'C:\\Projects\\cyber-security-awareness-platform\\backend\\reports\\ml\\android_malware\\data_review.csv', 'predictions': 'C:\\Projects\\cyber-security-awareness-platform\\backend\\reports\\ml\\andro

In [16]:
# Prepare audit tables in memory only; persisted outputs are written once at the end.
reason_by_row = []
for i, reasons in enumerate(row_reasons):
    if quarantine_mask[i]:
        reason_by_row.append(";".join(sorted(component_reasons[component_for_row[i]])))
    else:
        reason_by_row.append("")
component_size_by_id = {}
for indices in members.values():
    cid = component_for_row[indices[0]]
    component_size_by_id[cid] = len(indices)

manifest_columns = [
    "source_row_index", "source_dataset", "label_original", "target_interpretation_provisional",
    "package_id_for_grouping_only", "category_audit_only", "feature_vector_sha256",
    "component_id", "component_size", "partition", "review_required",
    "review_reason", "exclusion_reason", "exact_record_duplicate",
    "model_observation_representative",
]
review_rows = []
with ARTIFACT_PATHS["manifest"].open("x", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=manifest_columns)
    writer.writeheader()
    for i in range(len(row_ids)):
        record_key = record_key_by_index[i]
        is_representative = exact_record_rows[record_key][0] == i
        row = {
            "source_row_index": row_ids[i],
            "source_dataset": "android_benign_malware_dataset2",
            "label_original": labels_original[i],
            "target_interpretation_provisional": TARGET_NAMES[int(labels_array[i])],
            "package_id_for_grouping_only": packages[i],
            "category_audit_only": categories[i],
            "feature_vector_sha256": feature_hashes[i],
            "component_id": component_for_row[i],
            "component_size": component_size_by_id[component_for_row[i]],
            "partition": row_partition[i],
            "review_required": bool(review_flags[i]),
            "review_reason": "output_polarity_not_verified_from_dataset_documentation",
            "exclusion_reason": reason_by_row[i],
            "exact_record_duplicate": not is_representative,
            "model_observation_representative": is_representative and not quarantine_mask[i],
        }
        writer.writerow(row)
        if row["exclusion_reason"] or row["exact_record_duplicate"]:
            review_rows.append(row)
with ARTIFACT_PATHS["review"].open("x", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=manifest_columns)
    writer.writeheader()
    writer.writerows(review_rows)

prediction_frame.to_csv(ARTIFACT_PATHS["predictions"], index=False)
error_frame.to_csv(ARTIFACT_PATHS["errors"], index=False)
joblib.dump(logistic_model, ARTIFACT_PATHS["logistic_model"])
joblib.dump(random_forest_model, ARTIFACT_PATHS["random_forest_model"])

metrics_payload = {
    "target_mapping": {"0": TARGET_NAMES[0], "1": TARGET_NAMES[1],
                       "status": "provisional; externally corroborated but not publisher-confirmed"},
    "seed": SEED, "threshold": THRESHOLD,
    "provisional_full_dataset_class_distribution": {"counts": dict(overall_target_counts),
        "shares": {str(k): v / len(labels_array) for k, v in overall_target_counts.items()}},
    "versions": {"python": platform.python_version(), "numpy": np.__version__,
                 "pandas": pd.__version__, "scipy": scipy.__version__,
                 "scikit_learn": sklearn.__version__, "joblib": joblib.__version__},
    "feature_preparation": {
        "input_features": len(feature_columns), "constant_columns_removed": len(constant_columns),
        "duplicate_columns_beyond_first": duplicate_columns_beyond_first,
        "retained_feature_count": len(retained_features), "retained_features": retained_features,
        "package_and_category_excluded": True,
    },
    "alignment": {"workbook_rows": len(identity_by_row), "candidate_rows": len(row_ids),
                  "label_mismatches": len(alignment_mismatches), "verified_by_source_row_index": True},
    "duplicate_conflict_policy": {
        "exact_complete_duplicate_rows_collapsed": exact_duplicate_rows_beyond_first,
        "conflicting_package_ids": len(conflicting_packages),
        "conflicting_feature_vectors": len(conflicting_feature_hashes),
        "quarantined_components": len(quarantined_components),
        "quarantined_source_rows": int(quarantine_mask.sum()),
        "reasons_record_counts_overlap_possible": dict(reason_record_counts),
        "component_propagated_reason_counts_overlap_possible": propagated_exclusion_counts,
    },
    "components": {"count": len(members), "mixed_label_count": mixed_label_components,
                     "label_count_patterns": {str(k): v for k, v in component_label_pattern_counts.items()},
                     "size_median_p90_p95_max": {"median": percentile(component_sizes,.5),
                         "p90": percentile(component_sizes,.9), "p95": percentile(component_sizes,.95),
                         "max": max(component_sizes)}, "largest_sizes": sorted(component_sizes,reverse=True)[:10]},
    "split": {"method": "StratifiedGroupKFold with 20 folds; whole package/feature connected components",
              "fold_mapping": "14 train folds, 3 validation folds, 3 test folds",
              "record_counts_and_classes": split_summary,
              "component_isolation_verified": True, "package_isolation_verified": True,
              "feature_vector_isolation_verified": True, "quarantine_isolation_verified": True},
    "models": {
        "logistic_regression": {"selected_class_weight": selected_logistic_key,
                                "validation_candidates": logistic_validation,
                                "validation_metrics": lr_selected_validation_metrics,
                                "test_metrics": lr_test_metrics,
                                "config": {"C":1.0,"solver":"liblinear","max_iter":2000,"threshold":THRESHOLD}},
        "random_forest": {"validation_metrics":rf_validation_metrics,"test_metrics":rf_test_metrics,
                          "config":{"n_estimators":200,"max_features":"sqrt","min_samples_leaf":2,
                                    "class_weight":"balanced_subsample","random_state":SEED,"n_jobs":-1}},
    },
    "interpretation": "Dataset-source exploratory evaluation, not independent real-world accuracy. OUTPUT polarity is provisional.",
    "top_feature_importance": importance_rows,
    "artifacts": {key: str(value) for key, value in ARTIFACT_PATHS.items()},
}
with ARTIFACT_PATHS["config"].open("x", encoding="utf-8") as f:
    json.dump({"seed": SEED, "threshold": THRESHOLD, "target_mapping_note": TARGET_MAPPING_NOTE,
               "feature_columns": retained_features, "split": metrics_payload["split"],
               "models": {k:v["config"] if "config" in v else None for k,v in metrics_payload["models"].items()}},
              f, indent=2)
    f.write("\n")
with ARTIFACT_PATHS["metrics"].open("x", encoding="utf-8") as f:
    json.dump(metrics_payload, f, indent=2)
    f.write("\n")

report_lines = [
    "# Android malware classifier — exploratory grouped evaluation", "",
    "## Label provenance", TARGET_MAPPING_NOTE,
    "Labels are not independently verified; treat all metrics as provisional and dataset-source results.", "",
    "## Data preparation and exclusions",
    f"- Source records: {len(row_ids):,}; unique feature vectors: {len(set(feature_hashes)):,}.",
    f"- Provisional target counts: {dict(overall_target_counts)}; shares: {overall_target_shares}. Accuracy alone is insufficient under this imbalance.",
    f"- Conflicting package IDs: {len(conflicting_packages):,}; conflicting feature vectors: {len(conflicting_feature_hashes):,}.",
    f"- Quarantined components/rows: {len(quarantined_components):,} / {int(quarantine_mask.sum()):,}.",
    f"- Exact complete duplicate rows collapsed for modeling: {exact_duplicate_rows_beyond_first:,}.",
    f"- Package IDs repeated: {sum(count > 1 for count in package_row_counts.values()):,}; package IDs with multiple feature vectors: {sum(len(v) > 1 for v in package_feature_vectors.values()):,}.",
    f"- Features: {len(feature_columns):,} input, {len(constant_columns):,} constant removed, {duplicate_columns_beyond_first:,} duplicate beyond first, {len(retained_features):,} retained.", "",
    f"- Connected components: {len(members):,}; mixed-label components: {mixed_label_components:,}; component sizes median/p90/p95/max = {percentile(component_sizes,.5)}/{percentile(component_sizes,.9)}/{percentile(component_sizes,.95)}/{max(component_sizes)}.",
    "## Split", "Method: grouped StratifiedGroupKFold (seed 20261026); package and identical feature-vector components remain intact.",
    f"Partition summary: {json.dumps(split_summary, sort_keys=True)}", "",
    "## Test metrics",
    f"- Logistic Regression (class_weight={selected_logistic_key}): {json.dumps(lr_test_metrics, sort_keys=True)}",
    f"- Random Forest: {json.dumps(rf_test_metrics, sort_keys=True)}",
    comparison_interpretation,
    "Both models use the same held-out test components. Logistic class_weight was selected on validation F1, with recall as tie-break.", "",
    "False negatives represent missed malware under the provisional labels; false positives represent non-malware flagged as malware. Both carry operational cost.",
    "Feature importance is associational and does not prove that a permission causes malware.",
    "Top feature importance is included in metrics.json; inspect logistic coefficient directions and Random Forest impurity importance separately.",
    "This within-dataset result is limited by unconfirmed label polarity, repeated/identical feature vectors, package ambiguity, and collection bias.",
]
with ARTIFACT_PATHS["report"].open("x", encoding="utf-8") as f:
    f.write("\n".join(report_lines) + "\n")
print("Saved Android experiment artifacts:")
print(json.dumps({key: str(value) for key, value in ARTIFACT_PATHS.items()}, indent=2))
print("Metrics:", json.dumps(comparison, indent=2))

Saved Android experiment artifacts:
{
  "logistic_model": "C:\\Projects\\cyber-security-awareness-platform\\backend\\models\\android\\android_logistic_regression_grouped.joblib",
  "random_forest_model": "C:\\Projects\\cyber-security-awareness-platform\\backend\\models\\android\\android_random_forest_grouped.joblib",
  "config": "C:\\Projects\\cyber-security-awareness-platform\\backend\\reports\\ml\\android_malware\\experiment_config.json",
  "metrics": "C:\\Projects\\cyber-security-awareness-platform\\backend\\reports\\ml\\android_malware\\metrics.json",
  "report": "C:\\Projects\\cyber-security-awareness-platform\\backend\\reports\\ml\\android_malware\\experiment_report.md",
  "manifest": "C:\\Projects\\cyber-security-awareness-platform\\backend\\reports\\ml\\android_malware\\split_manifest.csv",
  "review": "C:\\Projects\\cyber-security-awareness-platform\\backend\\reports\\ml\\android_malware\\data_review.csv",
  "predictions": "C:\\Projects\\cyber-security-awareness-platform\\back

## Interpretation and next review

Review the model comparison, false positives/negatives, retained feature list, component allocation, and excluded records before treating the model as usable. Large group constraints or weak class representation should lead to more provenance/review work, not row-level split adjustments. Confirm OUTPUT polarity with the publisher or independent app-level label verification before claiming Android malware classification performance.